# RSNA 2026: End-to-End Multi-Fold SOTA Inference

**Architecture:** DINOv2-Base + Pre-LN Cross-Slot Transformer (5-Fold Ensemble)
**Resolution & Slices:** $518 \times 518$ Native DINOv2 Patches ($37 \times 37$ tokens), Depth $D=32$
**Inference Optimizations:**
- Automated `rsna.zip` / Dataset extraction & path resolution
- Inverted-cache 4-pass TTA (strictly zero horizontal flips to protect medial/lateral laterality)
- Dual-T4 GPU parallel acceleration & FP16/BF16 autocast
- Non-fatal missing series imputation (guaranteed submission generation)

---
### 📦 Instructions for Kaggle Submission
1. **Disable Internet** in your Kaggle Notebook settings (mandatory for submission).
2. **Attach Competition Data:** `rsna-knee-abnormality-detection`.
3. **Attach Model Artifacts:** Upload `rsna.zip` (containing `src/`, `pipeline_out/models_fold*`, and `configs/`) as a private Kaggle Dataset.
4. **(Optional) Attach Wheels:** If using `dicomsdl`, attach a wheel dataset to `/kaggle/input`.

In [ ]:
# ==============================================================================
# STEP 1: OFFLINE DEPENDENCY INSTALLATION
# ==============================================================================
import os
import sys
import glob

# Auto-detect and install any offline wheels (.whl) in /kaggle/input
wheels = glob.glob('/kaggle/input/**/*.whl', recursive=True)
if wheels:
    print(f'[SETUP] Found {len(wheels)} offline wheel packages. Installing...')
    for whl in wheels:
        os.system(f'pip install "{whl}" --no-index --find-links /kaggle/input/ --quiet')
else:
    print('[SETUP] No extra .whl files detected. Proceeding with pre-installed packages.')


In [ ]:
# ==============================================================================
# STEP 2: AUTO-EXTRACTION OF RSNA.ZIP & PATH DISCOVERY
# ==============================================================================
import zipfile
import shutil

WORKING_DIR = '/kaggle/working'
EXTRACT_DIR = os.path.join(WORKING_DIR, 'rsna_bundle')

# 1. Locate zip archive if uploaded as rsna.zip
zip_candidates = glob.glob('/kaggle/input/**/rsna*.zip', recursive=True) + glob.glob('/kaggle/input/**/*.zip', recursive=True)
if zip_candidates:
    target_zip = zip_candidates[0]
    print(f'[AUTO-DETECT] Found archive: {target_zip}')
    print(f'[AUTO-DETECT] Extracting to {EXTRACT_DIR}...')
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    with zipfile.ZipFile(target_zip, 'r') as zip_ref:
        zip_ref.extractall(EXTRACT_DIR)
    print('[SUCCESS] Archive extracted successfully.')
else:
    print('[INFO] No .zip archive found. Checking for unzipped dataset folders...')

# 2. Discover REPO_ROOT (where src/ is located)
candidate_repo_roots = [
    EXTRACT_DIR,
    os.path.join(EXTRACT_DIR, 'Knee_RSNA'),
    '/kaggle/working',
    '/kaggle/input/rsna-knee-models',
    '/kaggle/input/rsna-models',
    '/kaggle/input/rsna'
]
for path in glob.glob('/kaggle/input/**/src', recursive=True):
    candidate_repo_roots.insert(0, os.path.dirname(path))

REPO_ROOT = None
for cr in candidate_repo_roots:
    if cr and os.path.exists(os.path.join(cr, 'src', 'inference', 'inference.py')):
        REPO_ROOT = cr
        break

if REPO_ROOT is None:
    # Fallback search
    REPO_ROOT = EXTRACT_DIR if os.path.exists(EXTRACT_DIR) else '/kaggle/working'

print(f'[CONFIG] Active Codebase Root (REPO_ROOT): {REPO_ROOT}')
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

# 3. Discover WEIGHTS_DIR (where models_fold* are located)
candidate_weights_dirs = [
    os.path.join(REPO_ROOT, 'pipeline_out'),
    os.path.join(EXTRACT_DIR, 'pipeline_out'),
    REPO_ROOT,
    EXTRACT_DIR,
    '/kaggle/input/rsna-knee-models/pipeline_out',
    '/kaggle/input/rsna-knee-models'
]
for path in glob.glob('/kaggle/input/**/models_fold0', recursive=True) + glob.glob(f'{EXTRACT_DIR}/**/models_fold0', recursive=True):
    candidate_weights_dirs.insert(0, os.path.dirname(path))

WEIGHTS_DIR = None
for cw in candidate_weights_dirs:
    if cw and (os.path.exists(os.path.join(cw, 'models_fold0')) or os.path.exists(os.path.join(cw, 'fold0_ema.pt'))):
        WEIGHTS_DIR = cw
        break

print(f'[CONFIG] Active Weights Directory: {WEIGHTS_DIR}')


In [ ]:
# ==============================================================================
# STEP 3: LOAD 5-FOLD ENSEMBLE CHECKPOINTS
# ==============================================================================
import torch
import src.core.config as config
from src.modeling.model import load_checkpoint, build_model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
cfg = config.get_cfg('v2')
models = []

print(f'[CUDA] Target Device: {device} | Device Count: {torch.cuda.device_count()}')
print('Loading 5-Fold Trained Checkpoints...')

for fold in range(5):
    # Priority: 1) models_fold{f}/fold{f}_ema.pt, 2) models_fold{f}/fold{f}_best.pt, 3) fold{f}_ema.pt in root
    candidates = [
        os.path.join(WEIGHTS_DIR, f'models_fold{fold}', f'fold{fold}_ema.pt'),
        os.path.join(WEIGHTS_DIR, f'models_fold{fold}', f'fold{fold}_best.pt'),
        os.path.join(WEIGHTS_DIR, f'fold{fold}_ema.pt'),
        os.path.join(WEIGHTS_DIR, f'fold{fold}_best.pt')
    ] if WEIGHTS_DIR else []
    
    ckpt_path = None
    for cp in candidates:
        if os.path.exists(cp):
            ckpt_path = cp
            break
            
    if ckpt_path is None:
        print(f'[WARNING] Checkpoint for fold {fold} not found. Skipping fold {fold}.')
        continue
        
    print(f'[MODEL] Loading Fold {fold} from: {ckpt_path}')
    m = load_checkpoint(ckpt_path, device=device)
    models.append(m)

if not models:
    raise RuntimeError('FATAL: No trained model checkpoints could be loaded! Check your zip file or dataset.')

print(f'[SUCCESS] Successfully loaded {len(models)} model(s) into the ensemble.')


In [ ]:
# ==============================================================================
# STEP 4: EXECUTE INFERENCE & WRITE SUBMISSION
# ==============================================================================
import pandas as pd
from src.inference.inference import run_inference

# Locate test dataset
DATA_ROOT = None
for candidate_root in [
    '/kaggle/input/rsna-knee-abnormality-detection',
    '/kaggle/input/rsna-knee-challenge',
    '/kaggle/input/rsna-2026'
]:
    if os.path.exists(os.path.join(candidate_root, 'test.csv')) or os.path.exists(os.path.join(candidate_root, 'test_series.csv')):
        DATA_ROOT = candidate_root
        break

if DATA_ROOT is None:
    # Default fallback
    DATA_ROOT = '/kaggle/input/rsna-knee-abnormality-detection'

TEST_CSV = os.path.join(DATA_ROOT, 'test.csv')
OUT_CSV = '/kaggle/working/submission.csv'
CACHE_DIR = '/kaggle/temp' if os.path.isdir('/kaggle/temp') else '/kaggle/working/cache'

print(f'[DATA] Data Root: {DATA_ROOT}')
print(f'[DATA] Test CSV: {TEST_CSV}')
print(f'[EXEC] Running full ensemble inference (4-Pass TTA, Dual-GPU DataParallel)...')

sub, stats = run_inference(
    root=DATA_ROOT,
    models=models,
    test_csv=TEST_CSV,
    cfg=cfg,
    out_csv=OUT_CSV,
    cache_dir=CACHE_DIR,
    workers=4,
    chunk=64,
    device=device,
    use_tta=True,   # 4-Pass TTA without flips (+0.005 AUC)
    n_tta=4,
    batch=8         # Dual T4 saturation (4 studies/GPU)
)

# Clean up temporary cache to avoid Kaggle disk quota errors
if os.path.exists(CACHE_DIR):
    shutil.rmtree(CACHE_DIR, ignore_errors=True)
if os.path.exists(EXTRACT_DIR):
    # Remove heavy unzipped files if needed, keeping submission intact
    pass

print('\n' + '=' * 80)
print('[VERIFICATION] SUBMISSION INTEGRITY CHECK')
print('=' * 80)
print(f'Submission shape: {sub.shape}')
print(f'Null count: {sub.isnull().sum().sum()}')
print('Preview of submission.csv:')
print(sub.head(10))
print('\n[SUCCESS] submission.csv is ready for Kaggle scoring!')
